In [1]:
!pip install -q pytorch-lightning datasets transformers evaluate accelerate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 58.8 MB/s eta 0:00:00


In [2]:
# Instalacion de librerias
import random
import torch
import numpy as np
import os
from pytorch_lightning import seed_everything
import matplotlib.pyplot as plt
import seaborn as sns
import re

seed_val = 42
random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
torch.cuda.manual_seed_all(seed_val)# Store the average loss after eachepoch so we can plot them.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ["TF_DETERMINISTIC_OPS"] = "1" # See:https://github.com/NVIDIA/tensorflow-determinism#confirmed-current-gpu-specific-sources-of-non-determinism-with-solutions
seed_everything(42, workers=True)

from datasets import Dataset, DatasetDict #, load_metric EN PRINCIPIO ESTÁ DESCONTINUADO, TENDREMOS QUE BUSCAR OTRA ALTERNATIVA
import pandas as pd
import sklearn as sk
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score, f1_score
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, \
TrainingArguments, Trainer, pipeline, EarlyStoppingCallback
from huggingface_hub import login

INFO:lightning_fabric.utilities.seed:Seed set to 42


In [3]:
# Comprobación de disponibilidad de GPU
if torch.cuda.is_available():
    # Si hay una GPU disponible, la asignamos como dispositivo principal
    device = torch.device("cuda")
    print(f'✅ GPU detectada. Trabajando con: "{torch.cuda.get_device_name(0)}"')
else:
    # Si no hay GPU, el modelo y los tensores irán a la CPU
    device = torch.device("cpu")
    print('⚠️ Trabajando con CPU.')
    print('Para usar la GPU en Colab: Ve al menú superior "Entorno de ejecución" -> "Cambiar tipo de entorno de ejecución" -> Selecciona "GPU T4".')

# Opcional pero recomendado: Ver cuánta memoria VRAM tienes disponible
if device.type == 'cuda':
    memoria_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'Memoria VRAM total disponible: {memoria_total:.2f} GB')

✅ GPU detectada. Trabajando con: "Tesla T4"
Memoria VRAM total disponible: 15.64 GB


In [4]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

MessageError: Error: credential propagation was unsuccessful